In [67]:
import numpy as np
import pandas as pd
import os
import re
from numpy.linalg import norm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD
import plotly.express as px

In [68]:
LIB=pd.read_csv("Corpus_LIB.csv")

In [69]:
LIB

,book_title,author,book_len,n_chaps,avg_sent_len
0,The Old Curiosity Shop,Charles Dickens,218729,73,17.999424
1,The Mystery of Edwin Drood,Charles Dickens,96436,23,12.012456
2,Dombey and Son,Charles Dickens,356587,62,15.899189
3,Nicholas Nickleby,Charles Dickens,324259,65,14.234997
4,Bleak House,Charles Dickens,354643,67,13.623872
5,Martin Chuzzlewit,Charles Dickens,338838,55,13.576872
6,David Copperfield,Charles Dickens,357805,64,13.334513
7,A Tale of Two Cities 1,Charles Dickens,17520,6,15.077453
8,A Tale of Two Cities 2,Charles Dickens,70973,24,13.998619
9,A Tale of Two Cities 3,Charles Dickens,48583,15,13.394817


In [70]:
LIB["label"]=LIB["book_title"]

In [71]:
TFIDF=pd.read_csv("tfidf_L2.csv",index_col=0)

In [72]:
TFIDF

,1767,aaron,abbaye,abbey,abbeys,abel,accost,accused,ada,adas,...,wren,yaha,yarmouth,yer,yeth,yo,yod,yor,york,yourthelf
book_title,,,,,,,,,,,,,,,,,,,,,
A Tale of Two Cities 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.011963,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A Tale of Two Cities 2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.015423,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A Tale of Two Cities 3,0.011267,0.000000,0.013165,0.000000,0.000000,0.000000,0.000000,0.011689,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Barnaby Rudge,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Bleak House,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.307637,0.033439,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
David Copperfield,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.022441,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Dombey and Son,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Great Expectations,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Hard Times 1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.010706,0.000000,0.000000,0.000000,0.000000


In [73]:
pca_engine = PCA(n_components=5)
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF), index=TFIDF.index)
DCM.columns = ['PC{}'.format(i) for i in DCM.columns]
DCM

,PC0,PC1,PC2,PC3,PC4
book_title,,,,,
A Tale of Two Cities 1,-0.244568,-0.525786,-0.417915,0.213726,0.013131
A Tale of Two Cities 2,-0.261002,-0.573151,-0.466180,0.244026,-0.001399
A Tale of Two Cities 3,-0.256038,-0.558597,-0.453790,0.233295,-0.006248
Barnaby Rudge,-0.089076,-0.030975,0.097871,-0.252691,-0.495929
Bleak House,-0.084494,-0.029425,0.098266,-0.254698,0.358783
David Copperfield,-0.088020,-0.029414,0.098604,-0.253320,0.249239
Dombey and Son,-0.091006,-0.022137,0.093462,-0.245081,-0.081468
Great Expectations,-0.089645,-0.031947,0.096865,-0.253799,-0.434064
Hard Times 1,-0.316191,0.677992,-0.279733,0.203985,-0.002563


In [74]:
COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_))
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index = TFIDF.columns
COMPS.index.name = 'term_str'
COMPS.T

term_str,1767,aaron,abbaye,abbey,abbeys,abel,accost,accused,ada,adas,...,wren,yaha,yarmouth,yer,yeth,yo,yod,yor,york,yourthelf
PC0,-0.000363,0.002676,-0.000424,0.006217,0.001504,-0.000605,-0.000367,-0.000376,-0.003267,-0.000355,...,0.031301,-0.000506,-0.000248,0.000166,-0.000598,-0.003981,-0.000378,-0.000777,-0.000129,-0.000897
PC1,-0.000887,0.000183,-0.001036,0.000437,0.000104,-0.000234,-0.000887,-0.000920,-0.001276,-0.000139,...,0.002194,-0.001246,-0.000093,-0.000223,0.001447,0.009461,0.000888,0.001880,-0.000043,0.002170
PC2,-0.000776,-0.000361,-0.000907,-0.000887,-0.000209,0.000844,-0.000751,-0.000805,0.004570,0.000497,...,-0.004441,-0.001090,0.000335,0.000899,-0.000642,-0.004166,-0.000390,-0.000834,0.000167,-0.000963
PC3,0.000442,0.000410,0.000516,0.001135,0.000271,-0.002434,0.000438,0.000459,-0.013246,-0.001440,...,0.005478,0.000635,-0.000962,-0.002787,0.000532,0.003214,0.000285,0.000691,-0.000485,0.000798
PC4,-0.000039,0.000236,-0.000046,-0.000206,-0.000122,0.005302,0.000080,-0.000041,0.023374,0.002541,...,0.000723,-0.000028,0.001190,0.002742,-0.000008,0.000149,0.000029,-0.000011,-0.000590,-0.000013


In [75]:
def vis_pcs(DCM, a, b, label='book_title', hover_name='label'):
    return px.scatter(DCM, f"PC{a}", f"PC{b}", 
                    color=LIB[label], 
                    hover_name=LIB[hover_name], 
                    size=LIB['book_len'],
                    marginal_x='box', height=800)

In [76]:
def vis_loadings(COMPS, a=0, b=1, hover_name='term_str'):
    return px.scatter(COMPS.reset_index(), f"PC{a}", f"PC{b}", 
                      text='term_str', 
                      marginal_x='box', 
                      height=800)

In [77]:
vis_pcs(DCM, 0, 1)

In [78]:
vis_loadings(COMPS, 0, 1)

In [79]:
vis_pcs(DCM, 2,3)

In [80]:
vis_loadings(COMPS, 2, 3)

In [81]:
top_terms_sk = {}
data = []
for i in range(5):
    for j in [0, 1]:
        data.append((i, j, ' '.join(COMPS.sort_values(f'PC{i}', ascending=bool(j)).head(10).index.to_list())))
comp_strs = pd.DataFrame(data)
comp_strs.columns =  ['pc', 'pole', 'gloss']
comp_strs = comp_strs.set_index(['pc', 'pole'])
comp_strs.unstack()

gloss  \
pole                                                  0   
pc                                                        
0     boffin wegg bella fledgeby eugene lammle rider...   
1     bounderby gradgrind sparsit louisa sissy steph...   
2     dorrit clennam pancks merdle meagles arthur fl...   
3     dorrit clennam lorry bounderby defarge pancks ...   
4     quilp swiveller nell jarndyce micawber grewgio...   

                                                         
pole                                                  1  
pc                                                       
0     bounderby lorry gradgrind defarge sparsit loui...  
1     lorry defarge manette pross carton darnay luci...  
2     lorry defarge bounderby gradgrind manette pros...  
3     pecksniff pickwick quilp dombey oliver nichola...  
4     hugh barnaby pecksniff pip wemmick haredale ha...

In [ ]:
PCA_Components = pd.DataFrame(comp_strs.unstack())

gloss  \
pole                                                  0   
pc                                                        
0     boffin wegg bella fledgeby eugene lammle rider...   
1     bounderby gradgrind sparsit louisa sissy steph...   
2     dorrit clennam pancks merdle meagles arthur fl...   
3     dorrit clennam lorry bounderby defarge pancks ...   
4     quilp swiveller nell jarndyce micawber grewgio...   

                                                         
pole                                                  1  
pc                                                       
0     bounderby lorry gradgrind defarge sparsit loui...  
1     lorry defarge manette pross carton darnay luci...  
2     lorry defarge bounderby gradgrind manette pros...  
3     pecksniff pickwick quilp dombey oliver nichola...  
4     hugh barnaby pecksniff pip wemmick haredale ha...

In [83]:
DCM.to_csv("PCA_DCM.csv")
COMPS.to_csv("PCA_COMPS.csv")
PCA_Components.to_csv("PCA_Components.csv")